In [63]:
import re
import numpy as np
from typing import List, Optional
from collections import defaultdict
from editdistance import eval as distance

def del_parentheses(text):
    pattern = r"\([^()]*\)"
    return re.sub(pattern, "", text)

def del_space(text):
    pattern = r"\s+"
    return re.sub(pattern, " ", text).strip()

def del_numbering(text):
    pattern = r"^(?:\d+[\.\)、]?\s*[\-\—\–]?\s*)?"
    return re.sub(pattern, "", text)

def is_in(text, items, threshold):
    for i in items:
        if (distance(i.lower(), text.lower()) <= threshold):
            return True
    return False

def nearest(text, items):
    """ given the raw text name and all candidates, 
        return {movie_name:, min_edit_distance: , nearest_movie: }
    """
    # calculate the edit distance
    items = list(set(items))
    dists = [distance(text.lower(), i.lower()) for i in items]
    # find the nearest movie
    nearest_idx = np.argmin(dists)
    nearest_movie = items[nearest_idx]
    return {
        'movie_name': text, 
        'min_edit_distance': dists[nearest_idx], 
        'nearest_movie': nearest_movie
    }


def extract_movie_name(text):
    text = text.split('/')[-1]
    text = text.replace('_', ' ').replace('-', ' ').replace('>', ' ')
    return del_space(del_parentheses(text))

def recall_score(gt_list, pred_list, ks, threshold, verbose=False):
    hits = defaultdict(list)
    for gt, preds in zip(gt_list, pred_list):
        for k in ks:
            hits[k].append(int(is_in(gt, preds[:k], threshold)))
    if verbose:
        for k in ks:
            print("Recall@{}: {:.4f}".format(k, np.mean(hits[k])))
    return hits
    

def mrr_score(gt_list, pred_list, ks, threshold, verbose=False):
    mrrs = defaultdict(list)
    for gt, preds in zip(gt_list, pred_list):
        for k in ks:
            for i, p in enumerate(preds[:k]):
                if is_in(gt, [p], threshold):
                    mrrs[k].append(1 / (i + 1))
                else:
                    mrrs[k].append(0)
    if verbose:
        for k in ks:
            print("MRR@{}: {:.4f}".format(k, np.mean(mrrs[k])))
    return mrrs

def ndcg_score(gt_list, pred_list, ks, threshold, verbose=False):
    ndcgs = defaultdict(list)
    for gt, preds in zip(gt_list, pred_list):
        for k in ks:
            for i, p in enumerate(preds[:k]):
                if is_in(gt, [p], threshold):
                    ndcgs[k].append(1 / np.log2(i + 2))
                    break
    if verbose:
        for k in ks:
            print("NDCG@{}: {:.4f}".format(k, np.mean(ndcgs[k])))
    return ndcgs

In [64]:
import os
import sys 
import json
from jsonargparse import CLI
from tqdm import tqdm

sys.path.append('./')

DIR = os.getcwd()

def extract_movies_from_conversation(text: str, max_movies: int = 5) -> List[str]:
    """
    Extract movie titles from natural conversation text.
    """
    movies = []
    
    # Pattern 1: Movies with years in parentheses
    year_pattern = r'([A-Za-z][A-Za-z\s:&\'\-\.!,0-9]*?)\s*\((\d{4})\)'
    year_matches = re.findall(year_pattern, text)
    for movie, year in year_matches:
        movie = movie.strip()
        if len(movie) > 2 and movie not in movies:
            movies.append(f"{movie} ({year})")
    
    # Pattern 2: Common movie title indicators
    movie_indicators = [
        r'(?:movie|film|watch|seen?)\s+([A-Z][A-Za-z\s:&\'\-\.!,0-9]{2,30})(?:\s|$|\.|\,)',
        r'(?:like|similar to|recommend)\s+([A-Z][A-Za-z\s:&\'\-\.!,0-9]{2,30})(?:\s|$|\.|\,)',
        r'(?:^|\s)([A-Z][A-Za-z\s:&\'\-\.!,0-9]{2,30})\s+(?:is|was)\s+(?:a|an|good|great|amazing)',
    ]
    
    for pattern in movie_indicators:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            match = match.strip()
            if len(match) > 2 and match not in movies and len(movies) < max_movies:
                movies.append(match)
    
    # Pattern 3: Quoted titles
    quoted_pattern = r'["\']([A-Za-z][A-Za-z\s:&\'\-\.!,0-9]{2,30})["\']'
    quoted_matches = re.findall(quoted_pattern, text)
    for match in quoted_matches:
        match = match.strip()
        if len(match) > 2 and match not in movies and len(movies) < max_movies:
            movies.append(match)
    
    return movies[:max_movies]

def is_numbered_list(text: str) -> bool:
    """Check if text contains a numbered list of movies."""
    numbered_pattern = r'^\s*\d+[\.\)]\s+[A-Za-z]'
    lines = text.strip().split('\n')
    
    numbered_lines = 0
    for line in lines:
        if re.match(numbered_pattern, line.strip()):
            numbered_lines += 1
    
    return numbered_lines >= 2

def extract_list(l, candidates=None):
    """
    Extract movie list from text, handling both numbered lists and natural conversations.
    
    Args:
        l: Input text containing movies (either numbered list or conversation)
        candidates: Optional list of candidate movies for matching
        
    Returns:
        dict: {'rec_list': list of movie dicts, 'preference': preference text}
    """
    text = l
    result = {}
    
    # Check if this is a natural conversation without numbered list
    if not is_numbered_list(text):
        # Extract movies from conversation
        movies = extract_movies_from_conversation(text)
        
        if movies:
            # Use extracted movies
            rec_list = [del_numbering(del_space(del_parentheses(movie.strip()))) for movie in movies]
            preference = "Extracted from conversation"
        else:
            # Fallback to original logic if no movies found
            try:
                preference, text = text.split('1.', maxsplit=1)
            except Exception as e:
                print(f"No numbered list found and no movies extracted: {e}")
                preference = text
                text = ""
            
            if text:
                text = text.replace(',', '\n')
                rec_list = [del_numbering(del_space(del_parentheses(i.strip()))) for i in text.split('\n')]
            else:
                rec_list = []
    else:
        # Original logic for numbered lists
        try:
            preference, text = text.split('1.', maxsplit=1)
        except Exception as e:
            print(e)
            preference = ""
            text = text.replace(',', '\n')
        
        rec_list = [del_numbering(del_space(del_parentheses(i.strip()))) for i in text.split('\n')]
    
    # Filter out empty strings
    rec_list = [movie for movie in rec_list if movie.strip()]
    
    # Convert to the format expected by evaluation code
    if candidates is not None:
        # Use your original nearest function which returns a dict
        rec_list_with_matching = []
        for movie in rec_list:
            nearest_result = nearest(movie, candidates)
            rec_list_with_matching.append(nearest_result)  # Keep the full dict
        rec_list = rec_list_with_matching
    else:
        # If no candidates provided, create dict format with original movie names
        rec_list = [{'movie_name': movie, 'min_edit_distance': 0, 'nearest_movie': movie} for movie in rec_list]
    
    result['rec_list'] = rec_list
    result['preference'] = preference
    return result

In [65]:
dataset = 'redial'
model = "llama3-2-1b-instruct"
# get paths
alg = 'DPO'
file_name = '_test.jsonl' if alg == 'vanilla' else '_test_turn_entropy_lr2e-6.jsonl' 
pred_json = f'test_res/{alg}/{dataset}/{model}/{dataset}{file_name}'
gt_json = os.path.join(DIR, f'testsets/{dataset}/test.jsonl')
meta_json = os.path.join(DIR, f'testsets/{dataset}/entity2id.json')

print(pred_json)

# load pred_json
preds = [json.loads(l) for l in open(pred_json)]

# load gt_json
gts = [json.loads(l) for l in open(gt_json)][:len(preds)]
gts = {i: g for i, g in enumerate(gts)}

# load meta_json
name2id = json.load(open(meta_json))
id2name = {v: extract_movie_name(k) for k, v in name2id.items()}

# get candidates
candidates = list(id2name.values())
# pred_list = [extract_list(l, candidates) for l in tqdm(preds)]
from multiprocessing import Pool
from functools import partial

def parallel_extract(preds, candidates=None, num_processes=None):
    extract_with_candidates = partial(extract_list, candidates=candidates)
    with Pool(processes=num_processes) as pool:
        pred_list = list(tqdm(
            pool.imap(extract_with_candidates, preds),
            total=len(preds),
            desc="Processing"
        ))
    return pred_list

pred_list = parallel_extract(preds, candidates, num_processes=400)
# make sure the index of gts and preds are the same
pred_dict = {i: p for i, p in zip(range(len(pred_list)), pred_list)} 

# get gt_list and prev_list
for idx in gts:
    gt = gts[idx]
    gt_list = [id2name[r] for r in gt['rec']]
    prev_list = [id2name[r] for r in gt['prev_entity']]
    pred_dict[idx]['gt_list'] = gt_list
    pred_dict[idx]['prev_list'] = prev_list

# save the intermediate results
os.makedirs(os.path.join(DIR, f'test_res/{alg}/{dataset}/{model}/intermediate/'), exist_ok=True)

extracted_path = os.path.join(DIR, f'test_res/{alg}/{dataset}/{model}/intermediate/extracted.jsonl')
with open(extracted_path, 'w') as f:
    # sorted keys
    for idx in sorted(pred_dict.keys()):
        json.dump(pred_dict[idx], f)
        f.write('\n')

test_res/DPO/redial/llama3-2-1b-instruct/redial_test_turn_entropy_lr2e-6.jsonl


Processing:   0%|          | 1/3552 [00:00<22:13,  2.66it/s]

not enough values to unpack (expected 2, got 1)


Processing:   0%|          | 2/3552 [00:01<55:49,  1.06it/s]

not enough values to unpack (expected 2, got 1)


...

Processing:  64%|██████▎   | 2263/3552 [00:38<00:20, 62.31it/s]

No numbered list found and no movies extracted: not enough values to unpack (expected 2, got 1)


Processing:  71%|███████▏  | 2537/3552 [00:42<00:12, 78.72it/s]

No numbered list found and no movies extracted: not enough values to unpack (expected 2, got 1)
No numbered list found and no movies extracted: not enough values to unpack (expected 2, got 1)
not enough values to unpack (expected 2, got 1)


Processing: 100%|██████████| 3552/3552 [00:52<00:00, 67.09it/s] 


In [66]:
! python evaluate.py --from_json {extracted_path}

4778it [00:00, 462570.04it/s]
4778it [00:00, 515203.47it/s]
4778it [00:00, 489637.78it/s]
4778it [00:00, 536125.86it/s]
Results are saved in /home/sagemaker-user/csbai/multiturn_rl/evaluation/zeroshot_test/test_res/DPO/redial/llama3-2-1b-instruct/intermediate!


In [67]:
import pandas as pd

# Replace 'your_file.csv' with the actual path to your CSV file
df = pd.read_csv(f'test_res/{alg}/{dataset}/{model}/intermediate/filtered_True_exclude_seen_True/summary.csv')
print("Model: "+ model+" on Dataset: "+dataset)
print("recall@1_mean", df['recall@1_mean'][0])
print("recall@1_se", df['recall@1_se'][0])
print("recall@5_mean", df['recall@5_mean'][0])
print("recall@5_se", df['recall@5_se'][0])
# print("recall@10_mean", df['recall@10_mean'][0])
# print("recall@10_se", df['recall@10_se'][0])
# print("recall@20_mean", df['recall@20_mean'][0])
# print("recall@20_se", df['recall@20_se'][0])

Model: llama3-2-1b-instruct on Dataset: redial
recall@1_mean 0.019356343283582
recall@1_se 0.0021042170138772
recall@5_mean 0.0494402985074626
recall@5_se 0.0033109566886774
